# Мини-Transformer
## Задача: по нескольким предыдущим символам предсказать следующий символ.

In [1]:
import random
import math

In [2]:
D_MODEL = 8       # размер вектора каждого символа
D_HIDDEN = 16     # размер скрытого слоя feed-forward
SEQ_LEN = 12      # сколько символов модель видит одновременно
EPOCHS = 100      # сколько раз пройти по обучающему тексту
LEARNING_RATE = 0.02

random.seed(1)

In [3]:
TEXT = (
    "магистрант изучает трансформер. "
    "трансформер использует внимание. "
) * 12

In [4]:
# Словарь символов
chars = sorted(set(TEXT))
VOCAB_SIZE = len(chars)

char_to_id = {}
id_to_char = {}

for i, ch in enumerate(chars):
    char_to_id[ch] = i
    id_to_char[i] = ch

In [5]:
# 3. Простые операции с векторами

def zeros(rows, cols): # Создаёт нулевую матрицу размером
    return [[0.0 for _ in range(cols)] for _ in range(rows)]


def random_matrix(rows, cols): # Создаёт матрицу rows x cols со случайными числами от -0.3 до 0.3
    matrix = zeros(rows, cols)

    for i in range(rows):
        for j in range(cols):
            matrix[i][j] = random.uniform(-0.3, 0.3)

    return matrix


def matvec(matrix, vector):
    # Умножает матрицу на вектор и возвращает новый вектор
    result = []

    for row in matrix:
        value = 0.0

        for i in range(len(vector)):
            value += row[i] * vector[i]

        result.append(value)

    return result


def add_vectors(a, b):
    # Складывает два вектора поэлементно
    return [a[i] + b[i] for i in range(len(a))]


def softmax(values):
    # Преобразует список чисел в вероятности (сумма равна 1)
    max_value = max(values)
    exp_values = []

    for value in values:
        exp_values.append(math.exp(value - max_value))

    total = sum(exp_values)
    return [value / total for value in exp_values]


def tanh(value):
    # гиперболический тангенс числа
    return math.tanh(value)

In [6]:
# 4. Параметры Transformer (кодирование во внутр. предст. и обратно)

# Эмбеддинги символов: VOCAB_SIZE x D_MODEL
E = random_matrix(VOCAB_SIZE, D_MODEL)

# Позиционные коды: SEQ_LEN x D_MODEL
P = random_matrix(SEQ_LEN, D_MODEL)

# Feed-forward слой:
# W1 переводит D_MODEL -> D_HIDDEN
W1 = random_matrix(D_HIDDEN, D_MODEL)
B1 = [0.0] * D_HIDDEN

# W2 переводит D_HIDDEN -> D_MODEL
W2 = random_matrix(D_MODEL, D_HIDDEN)
B2 = [0.0] * D_MODEL

# Выходной слой: D_MODEL -> VOCAB_SIZE
WO = random_matrix(VOCAB_SIZE, D_MODEL)
BO = [0.0] * VOCAB_SIZE

| Часть кода                         | Роль в attention                                             |
| ---------------------------------- | ------------------------------------------------------------ |
| X[t]                               | Query текущего символа: «что я ищу?»                         |
| X[j]                               | Key предыдущего символа: «что я могу предложить?»            |
| score += X[t][d] * X[j][d]         | Скалярное произведение Query и Key - оценка важности         |
| / math.sqrt(D_MODEL)               | Масштабирование, чтобы softmax не был слишком «острым»       |
| weights = softmax(scores)          | Превращает оценки в веса внимания, сумма которых равна 1     |
| context[d] += weights[j] * X[j][d] | Взвешенная сумма Value - новая информация с учётом контекста |
| Каузальная маска range(t + 1)      | Текущий символ не может «подглядывать» в будущее             |

In [7]:
# 5. Прямой проход Transformer

def forward(token_ids):
    """
    token_ids — последовательность ID символов.

    Возвращает:
    - logits: оценки для каждого возможного следующего символа
    - cache: промежуточные значения для обучения
    """

    X = []

    # Эмбеддинг символа + информация о позиции
    for position, token_id in enumerate(token_ids):
        x = add_vectors(E[token_id], P[position])
        X.append(x)

    # Self-attention с каузальной маской:
    # символ на позиции t видит только предыдущие символы.
    attention = []
    attention_weights = []

    for t in range(len(X)):
        scores = []

        for j in range(t + 1):
            score = 0.0

            for d in range(D_MODEL):
                score += X[t][d] * X[j][d]

            scores.append(score / math.sqrt(D_MODEL))

        weights = softmax(scores)
        attention_weights.append(weights)

        context = [0.0] * D_MODEL

        for j in range(t + 1):
            for d in range(D_MODEL):
                context[d] += weights[j] * X[j][d]

        attention.append(context)

    # Остаточное соединение после attention:
    # H = X + Attention(X)
    H = []

    for t in range(len(X)):
        H.append(add_vectors(X[t], attention[t]))

    # Feed-forward слой
    F1 = []
    F2 = []

    for t in range(len(X)):
        # D_MODEL -> D_HIDDEN
        hidden = matvec(W1, H[t])

        for i in range(D_HIDDEN):
            hidden[i] = tanh(hidden[i] + B1[i])

        # D_HIDDEN -> D_MODEL
        output = matvec(W2, hidden)

        for i in range(D_MODEL):
            output[i] += B2[i]

        F1.append(hidden)
        F2.append(output)

    # Остаточное соединение после feed-forward:
    # Y = H + FFN(H)
    Y = []

    for t in range(len(X)):
        Y.append(add_vectors(H[t], F2[t]))

    # Предсказание следующего символа
    logits = []

    for t in range(len(X)):
        logit = matvec(WO, Y[t])

        for i in range(VOCAB_SIZE):
            logit[i] += BO[i]

        logits.append(logit)

    cache = {
        "token_ids": token_ids,
        "X": X,
        "attention_weights": attention_weights,
        "H": H,
        "F1": F1,
        "Y": Y,
        "logits": logits
    }

    return logits, cache


In [8]:
# 6. Обратное распространение ошибки

def backward(cache, target_ids):
    """
    target_ids — правильные следующие символы.

    Возвращает градиенты всех обучаемых параметров.
    """

    X = cache["X"]
    attention_weights = cache["attention_weights"]
    H = cache["H"]
    F1 = cache["F1"]
    Y = cache["Y"]
    logits = cache["logits"]
    token_ids = cache["token_ids"]

    dE = zeros(VOCAB_SIZE, D_MODEL)
    dP = zeros(SEQ_LEN, D_MODEL)

    dW1 = zeros(D_HIDDEN, D_MODEL)
    dB1 = [0.0] * D_HIDDEN

    dW2 = zeros(D_MODEL, D_HIDDEN)
    dB2 = [0.0] * D_MODEL

    dWO = zeros(VOCAB_SIZE, D_MODEL)
    dBO = [0.0] * VOCAB_SIZE

    sequence_length = len(token_ids)
    count = sequence_length - 1

    # Градиенты по входам attention/feed-forward
    dX = [[0.0] * D_MODEL for _ in range(sequence_length)]

    for t in range(sequence_length - 1):
        target = target_ids[t]
        probabilities = softmax(logits[t])

        # Градиент cross-entropy
        dlogits = []

        for i in range(VOCAB_SIZE):
            if i == target:
                dlogits.append(probabilities[i] - 1.0)
            else:
                dlogits.append(probabilities[i])

        for i in range(VOCAB_SIZE):
            dlogits[i] /= count

        # Градиенты выходного слоя
        for d in range(D_MODEL):
            for i in range(VOCAB_SIZE):
                dWO[i][d] += Y[t][d] * dlogits[i]

        for i in range(VOCAB_SIZE):
            dBO[i] += dlogits[i]

        dY = [0.0] * D_MODEL

        for d in range(D_MODEL):
            value = 0.0

            for i in range(VOCAB_SIZE):
                value += WO[i][d] * dlogits[i]

            dY[d] = value

        # Градиент feed-forward
        dH = dY[:]
        dF2 = dY[:]

        for d in range(D_MODEL):
            dB2[d] += dF2[d]

        dF1 = [0.0] * D_HIDDEN

        # dF1 = W2^T * dF2, умноженный на производную tanh
        for h in range(D_HIDDEN):
            value = 0.0

            for d in range(D_MODEL):
                value += W2[d][h] * dF2[d]

            dF1[h] = value * (1.0 - F1[t][h] * F1[t][h])

        # dW2 = dF2 * F1^T
        for d in range(D_MODEL):
            for h in range(D_HIDDEN):
                dW2[d][h] += F1[t][h] * dF2[d]

        for h in range(D_HIDDEN):
            dB1[h] += dF1[h]

        # dW1 = dF1 * H^T
        for h in range(D_HIDDEN):
            for d in range(D_MODEL):
                dW1[h][d] += H[t][d] * dF1[h]

        # dH += W1^T * dF1
        for d in range(D_MODEL):
            value = 0.0

            for h in range(D_HIDDEN):
                value += W1[h][d] * dF1[h]

            dH[d] += value

        # Остаточное соединение:
        # H = X + attention, поэтому градиент идёт и в X, и в attention.
        for d in range(D_MODEL):
            dX[t][d] += dH[d]

        # Обратный проход через self-attention
        weights = attention_weights[t]

        # Градиент для Value:
        # attention[t] = сумма weights[j] * X[j]
        for j in range(t + 1):
            for d in range(D_MODEL):
                dX[j][d] += weights[j] * dH[d]

        # Градиенты для Query и Key
        for k in range(t + 1):
            dot_gx = 0.0

            for d in range(D_MODEL):
                dot_gx += dH[d] * X[k][d]

            weighted_sum = 0.0

            for j in range(t + 1):
                dot_gxj = 0.0

                for d in range(D_MODEL):
                    dot_gxj += dH[d] * X[j][d]

                weighted_sum += weights[j] * dot_gxj

            dweight = weights[k] * (dot_gx - weighted_sum)
            scale = 1.0 / math.sqrt(D_MODEL)

            # X[t] выступает как Query
            for d in range(D_MODEL):
                dX[t][d] += dweight * X[k][d] * scale

            # X[k] выступает как Key
            for d in range(D_MODEL):
                dX[k][d] += dweight * X[t][d] * scale

    # Градиенты эмбеддингов и позиционных кодов
    for t in range(sequence_length):
        token_id = token_ids[t]

        for d in range(D_MODEL):
            dE[token_id][d] += dX[t][d]
            dP[t][d] += dX[t][d]

    gradients = {
        "E": dE,
        "P": dP,
        "W1": dW1,
        "B1": dB1,
        "W2": dW2,
        "B2": dB2,
        "WO": dWO,
        "BO": dBO
    }

    return gradients


In [9]:
# 7. Обновление весов

def update_parameters(gradients):
    parameters = [
        (E, gradients["E"]),
        (P, gradients["P"]),
        (W1, gradients["W1"]),
        (W2, gradients["W2"]),
        (WO, gradients["WO"])
    ]

    for parameter, gradient in parameters:
        for i in range(len(parameter)):
            for j in range(len(parameter[0])):
                parameter[i][j] -= LEARNING_RATE * gradient[i][j]

    biases = [
        (B1, gradients["B1"]),
        (B2, gradients["B2"]),
        (BO, gradients["BO"])
    ]

    for bias, gradient in biases:
        for i in range(len(bias)):
            bias[i] -= LEARNING_RATE * gradient[i]

In [10]:
# 8. Подготовка обучающих примеров

def make_training_examples():
    inputs = []
    targets = []

    token_ids = [char_to_id[ch] for ch in TEXT]

    for start in range(0, len(token_ids) - SEQ_LEN):
        input_ids = token_ids[start:start + SEQ_LEN]
        target_ids = token_ids[start + 1:start + SEQ_LEN + 1]

        inputs.append(input_ids)
        targets.append(target_ids)

    return inputs, targets

In [11]:
# 9. Обучение
def train():
    inputs, targets = make_training_examples()

    print("Размер словаря:", VOCAB_SIZE)
    print("Число обучающих примеров:", len(inputs))
    

    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0

        for i in range(len(inputs)):
            logits, cache = forward(inputs[i])
            gradients = backward(cache, targets[i])
            update_parameters(gradients)

            # Ошибка считается только для предсказаний следующего символа
            for t in range(SEQ_LEN - 1):
                target = targets[i][t]
                probabilities = softmax(logits[t])
                total_loss -= math.log(probabilities[target] + 1e-12)

        average_loss = total_loss / (len(inputs) * (SEQ_LEN - 1))

        if epoch == 1 or epoch % 25 == 0:
            print(
                "Эпоха {0:3d} | Средняя ошибка: {1:.4f}".format(
                    epoch,
                    average_loss
                )
            )

    print("\nОбучение завершено.\n")

In [12]:
# 10. Генерация текста

def generate(start_text, length=80):
    result = start_text

    for _ in range(length):
        context = result[-SEQ_LEN:]

        token_ids = []

        for ch in context:
            if ch in char_to_id:
                token_ids.append(char_to_id[ch])
            else:
                token_ids.append(0)

        while len(token_ids) < SEQ_LEN:
            token_ids.insert(0, 0)

        logits, cache = forward(token_ids)
        probabilities = softmax(logits[-1])

        best_id = 0
        best_probability = -1.0

        for i in range(VOCAB_SIZE):
            if probabilities[i] > best_probability:
                best_probability = probabilities[i]
                best_id = i

        result += id_to_char[best_id]

    return result


In [14]:
train()

Размер словаря: 20
Число обучающих примеров: 768
Эпоха   1 | Средняя ошибка: 2.7062
Эпоха  25 | Средняя ошибка: 0.3033
Эпоха  50 | Средняя ошибка: 0.2321
Эпоха  75 | Средняя ошибка: 0.2066
Эпоха 100 | Средняя ошибка: 0.1992

Обучение завершено.



### Генерация после 100 эпох:

In [15]:
print("Пример генерации:")
print(generate("магистрант ", 80))

Пример генерации:
магистрант вннт трансание внсанимание трансфор внсансантаниансфоранспор испор тр тр тран вн


In [16]:
print(generate("трансформер ", 80))

трансформер тр тр тран внсансанимание вние таниманисфоранспор тр испор тр тр тран внсансаним


In [17]:
EPOCHS = 300 
train()

Размер словаря: 20
Число обучающих примеров: 768
Эпоха   1 | Средняя ошибка: 0.1989
Эпоха  25 | Средняя ошибка: 0.1878
Эпоха  50 | Средняя ошибка: 0.1861
Эпоха  75 | Средняя ошибка: 0.1739
Эпоха 100 | Средняя ошибка: 0.1734
Эпоха 125 | Средняя ошибка: 0.1671
Эпоха 150 | Средняя ошибка: 0.1652
Эпоха 175 | Средняя ошибка: 0.1869
Эпоха 200 | Средняя ошибка: 0.1606
Эпоха 225 | Средняя ошибка: 0.1588
Эпоха 250 | Средняя ошибка: 0.1577
Эпоха 275 | Средняя ошибка: 0.1569
Эпоха 300 | Средняя ошибка: 0.1564

Обучение завершено.



### Генерация после 300 эпох:

In [22]:
print(generate("магистрант ", 80))

магистрант внтан тан тан вн внананиананиантантантан тран вн тан танмантантан нсфоранспор вн


In [23]:
print(generate("трансформер ", 80))

трансформер тр тр тр вн внананананиантантантан тран вн тан танмантантан нсфоранспор внтантан
